# Create overlay masks

> Overlay mask creation from images and masks

In [ ]:
#| default_exp overlay_mask

In [ ]:
#| export
import pandas as pd
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from fastcore.all import *
import matplotlib as mpl
import matplotlib.pyplot as plt
from fastcore.all import *
import shutil
import cv2
from typing import Union, List, Tuple, Dict
import pandas as pd
from skimage import io, morphology, measure
from scipy.ndimage import (label, sum, binary_fill_holes)
import os
from tqdm.auto import tqdm
from argparse import ArgumentParser

In [ ]:
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
sys.path.append(str(CV_TOOLS))

# %% ../../../nbs/39_preprocessing.zero_degree_solder_pin.ipynb 6
custom_lib_path = Path(r'/home/ai_warstein/homes/goni/custom_libs')
sys.path.append(str(custom_lib_path))
from dotenv import load_dotenv

In [ ]:
CURRENT_REP = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/new_test')
sys.path.append(str(CURRENT_REP))

In [ ]:
#| exporti
from new_test.viz_utils.show_images import *

In [ ]:
#| export
from cv_tools.core import *

In [ ]:
#| export
def show_three_images(
    im1:np.ndarray,  # First image array
    im2:np.ndarray,  # Second image array
    
    title1:str='Image 1',  # Title for first image
    title2:str='Image 2',  # Title for second image
    
    figsize:tuple=(20,7),  # Larger figure size for better clarity
    layout:str='row',  # Layout type: 'row' or 'column'
    scale:float=1.0  # Scale factor to resize images
):
    'Show three images in a single row or column with titles, optimized for clarity'
    plt.figure(figsize=figsize, dpi=150)  # Higher DPI for better resolution
    
    # Scale images if needed
    if scale != 1.0:
        im1 = im1 * scale
        im2 = im2 * scale
        im3 = im3 * scale
    
    if layout == 'row':
        plt.subplot(121)
        plt.imshow(im1)
        plt.title(title1, pad=10, fontsize=12)
        plt.axis('off')
        
        plt.subplot(122)
        plt.imshow(im2)
        plt.title(title2, pad=10, fontsize=12)
        plt.axis('off')
        
     
    else:  # column layout
        plt.subplot(211)
        plt.imshow(im1)
        plt.title(title1, pad=10, fontsize=12)
        plt.axis('off')
        
        plt.subplot(212)
        plt.imshow(im2)
        plt.title(title2, pad=10, fontsize=12)
        plt.axis('off')
       
    plt.tight_layout()
    plt.show()


In [ ]:
#| eval: False
idx = 0
MODEL2='time_11_04_21_val_frGrnd0.9498_epoch_200.h5'
MODEL0='ETPD-WAR1_03.keras'
MODEL1='ETPD-WAR1_05.keras'
im_path = Path(r'/home/ai_easypid.work/CurrentTrainingData20240812_trn_val/Incoming_1B_Loetstift/Incoming_1B_Loetstift_unzip/main_im2_cropped')
failed_im_path = Path(im_path.parent, f'main_im2_cropped_masks/{MODEL2}/failed/missing/overlay')
sm_img_name = failed_im_path.ls()[idx].name
failed_img = Path(failed_im_path, sm_img_name)
#===================================================
if Path(im_path.parent, f'main_im2_cropped_masks/{MODEL0}/passed/overlay/{sm_img_name}').exists():
    model0_img = Path(im_path.parent, f'main_im2_cropped_masks/{MODEL0}/passed/overlay/{sm_img_name}')
    case0='passed'
elif Path(im_path.parent, f'main_im2_cropped_masks/{MODEL0}/failed/overlay/{sm_img_name}').exists():
    model0_img = Path(im_path.parent, f'main_im2_cropped_masks/{MODEL0}/failed/additional/overlay/{sm_img_name}')
    case0='additional'
else:
    model0_img = Path(im_path.parent, f'main_im2_cropped_masks/{MODEL0}/failed/missing/overlay/{sm_img_name}')
    case0='missing'
#===================================================
if Path(im_path.parent, 'main_im2_cropped_masks/{MODEL1}/passed/overlay/{sm_img_name}').exists():
    model1_img = Path(im_path.parent, f'main_im2_cropped_masks/{MODEL1}/passed/overlay/{sm_img_name}')
    case1='passed'
elif Path(im_path.parent, 'main_im2_cropped_masks/{MODEL0}/failed/overlay/{sm_img_name}').exists():
    model1_img = Path(im_path.parent, f'main_im2_cropped_masks/{MODEL1}/failed/additional/overlay/{sm_img_name}')
    case1='additional'
else:
    model1_img = Path(im_path.parent, f'main_im2_cropped_masks/{MODEL1}/failed/missing/overlay/{sm_img_name}')
    case1='missing'
#

In [ ]:
print(sm_img_name)
print(case0,case1)
case0=None 
case1=None

In [ ]:
show_three_images(
im1=read_img(failed_img, gray=False), im2=read_img(model0_img, gray=False), 
title1='New', title2='v3', figsize=(25,10), layout='column')

In [ ]:
#| export
def create_overlay_image(
        im_path:Union[str, Path],
        msk_path:Union[str,Path],
        overlay_mask_save_path:Union[str, Path],
        pr_im_path:Union[str, Path]=None
    ):
    'Create a overlay mask from image and mask path'
    if pr_im_path is not None:
        for i in tqdm(im_path.ls(),total=len(im_path.ls())):
            name_ = Path(i).name
            # process image has a name in processed_image and image2 has a name of image2 
            # so i am replacing image2 with processed_image
            pr_name = name_.replace('image2', 'processed_image')
            # reading processed image
            pr_img = read_img(f'{pr_im_path}/{pr_name}',gray=False )

            overlay_mask_border_on_image(
                im_path=i,
                msk_path=f'{msk_path}/{name_}',
                new_img=pr_img,
                save_overlay_img_path=overlay_mask_save_path
            )
    else:
        image_names = set(get_name_(im_path.ls())).intersection(set(get_name_(msk_path.ls())))
        for i in tqdm(image_names,total=len(image_names)):
            name_ = Path(i).name

            overlay_mask_border_on_image(
                im_path=f'{im_path}/{name_}',
                msk_path=f'{msk_path}/{name_}',
                save_overlay_img_path=overlay_mask_save_path
            )

In [ ]:
#| export
def parse_args_():
    parser = ArgumentParser(
                            )
    parser.add_argument(
       '--im_path', 
       type=str,
       help='Path to the images'
    )
    parser.add_argument(
        '--msk_path',
        type=str,
        help='Path to the masks'
    )
    parser.add_argument(
        '--overlay_mask_save_path',
        type=str,
        help='Path to save the overlay mask'
    )
    parser.add_argument(
        '--pr_image_path',
        default=None,
        help='Whether process image is available or not, in case of not available nothing will happen'
    )
    return parser.parse_args()

In [ ]:
#| export
def main_():
    args = parse_args_()
    create_overlay_image(
        im_path=Path(args.im_path),
        msk_path=Path(args.msk_path),
        pr_im_path=args.pr_image_path,
        overlay_mask_save_path=Path(args.overlay_mask_save_path)
    )

In [ ]:
#| export
if __name__ == '__main__':
    main_()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('06_create_overlay_masks.ipynb')